# Train RGCN v1: HC3 → GPT-3.5 Mixed (0% noise), batch 64

Notebook này train **RGCN v1** (`RGCN2` trong `rgcn.py`), không dùng V3 và không dùng R-Drop. Mô hình train trên HC3 train/validation, checkpoint được chọn theo validation accuracy của HC3, sau đó chỉ test **GPT-3.5 Mixed clean (0% noise)**.

`--batch-size 64` được dùng cho cả train và test. Graph GPT-3.5 chỉ được tạo sau khi train hoàn tất; RGCN v1 dùng một forward BCE cho mỗi batch.

In [ ]:
# Kaggle Python 3.13 needs current binary wheels. Do not pin transformers==4.44.2.
!pip install -q --upgrade torch-geometric transformers spacy scikit-learn tqdm
!python -m spacy download en_core_web_sm

import torch
import torch_geometric
import transformers
import spacy
print({'torch': torch.__version__, 'torch_geometric': torch_geometric.__version__,
       'transformers': transformers.__version__, 'spacy': spacy.__version__})

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

WORK = Path('/kaggle/working/group_rgcn_v1_hc3_to_gpt35_bs64')
REPO = WORK / 'PRDetect-Research'
BRANCH = 'cleanup/v3-core-only'
SEED = 2026
BATCH_SIZE = 64
EPOCHS = 40
TEST_SPLIT = 'gpt3.5_mixed_test_split'
RESULT = WORK / 'result'
CHECKPOINT = RESULT / f'hc3_rgcn_v1_batch{BATCH_SIZE}_seed{SEED}.pth'
if REPO.exists():
    shutil.rmtree(REPO)
WORK.mkdir(parents=True, exist_ok=True)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
                'https://github.com/tphan1420/PRDetect-Research.git', str(REPO)], check=True)
RESULT.mkdir(exist_ok=True)
assert (REPO / 'rgcn.py').is_file() and (REPO / 'test_rgcn.py').is_file()
assert (REPO / 'data/original_text/benchmarks/hc3/hc3_train.json').is_file()
assert (REPO / 'data/original_text/benchmarks/hc3/hc3_val.json').is_file()
assert (REPO / 'data/original_text/benchmarks/gpt35_mixed/gpt3.5_mixed_test_split.json').is_file()
assert not (REPO / 'graph_data' / f'{TEST_SPLIT}.pkl').exists(), 'Test graph must not exist before training.'
COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print({'commit': COMMIT, 'model': 'RGCN v1 / RGCN2',
       'seed': SEED, 'batch_size': BATCH_SIZE, 'test_only': TEST_SPLIT})

In [ ]:
# Build only HC3 training/validation graphs. They retain all_edge_type required by RGCN v1.
subprocess.run([sys.executable, 'building_graph.py', '--file', 'hc3_train', 'hc3_val'], cwd=REPO, check=True)
TRAIN_GRAPH = REPO / 'graph_data/hc3_train.pkl'
VAL_GRAPH = REPO / 'graph_data/hc3_val.pkl'
assert TRAIN_GRAPH.is_file() and VAL_GRAPH.is_file()
assert not (REPO / 'graph_data' / f'{TEST_SPLIT}.pkl').exists(), 'Test graph was created before training.'

In [ ]:
# RGCN v1 baseline: two layers, 30 bases, one BCE forward per batch (no R-Drop).
subprocess.run(
    [sys.executable, 'rgcn.py', '--dataset', 'hc3', '--train_file', 'hc3_train', '--val_file', 'hc3_val',
     '--seed', str(SEED), '--batch_size', str(BATCH_SIZE), '--epochs', str(EPOCHS),
     '--lr', '0.0001', '--weight_decay', '0.00001', '--hidden_dim', '256', '--output_dim', '64',
     '--num_layers', '2', '--num_bases', '30', '--dropout', '0.5', '--patience', '5',
     '--graph_dir', str(REPO / 'graph_data'), '--result_dir', str(RESULT),
     '--save_path', str(CHECKPOINT), '--no_test'],
    cwd=REPO,
    check=True,
)
TRAIN_METADATA = Path(str(CHECKPOINT) + '.train.json')
assert CHECKPOINT.is_file() and TRAIN_METADATA.is_file()
metadata = json.loads(TRAIN_METADATA.read_text(encoding='utf-8'))
assert metadata['method'] == 'baseline'
assert metadata['arguments']['train_file'] == 'hc3_train' and metadata['arguments']['val_file'] == 'hc3_val'

In [ ]:
# Build and evaluate exactly one post-training test split: GPT-3.5 Mixed 0%.
subprocess.run([sys.executable, 'building_graph.py', '--file', TEST_SPLIT], cwd=REPO, check=True)
GRAPH = REPO / 'graph_data' / f'{TEST_SPLIT}.pkl'
assert GRAPH.is_file()
REPORT = RESULT / 'gpt35_0pct_report.json'
subprocess.run(
    [sys.executable, 'test_rgcn.py', '--dataset', 'hc3', '--seed', str(SEED), '--file', TEST_SPLIT,
     '--batch_size', str(BATCH_SIZE), '--num_layers', '2', '--num_bases', '30',
     '--checkpoint', str(CHECKPOINT), '--graph_dir', str(REPO / 'graph_data'),
     '--result_dir', str(RESULT), '--report_json', str(REPORT), '--save'],
    cwd=REPO,
    check=True,
)
assert REPORT.is_file()
report = json.loads(REPORT.read_text(encoding='utf-8'))
assert report['split'] == TEST_SPLIT and report['samples'] == 1000 and report['threshold'] == 0.5

In [ ]:
import zipfile

RUN_CONFIG = RESULT / 'run_config.json'
RUN_CONFIG.write_text(json.dumps({
    'commit': COMMIT, 'branch': BRANCH, 'model': 'RGCN v1 / RGCN2',
    'seed': SEED, 'batch_size': BATCH_SIZE, 'epochs': EPOCHS,
    'train_split': 'hc3_train', 'validation_split': 'hc3_val', 'test_split': TEST_SPLIT,
    'checkpoint': str(CHECKPOINT), 'test_graph': str(GRAPH),
}, indent=2), encoding='utf-8')
archive = WORK / 'rgcn_v1_hc3_to_gpt35_0pct_batch64_results.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as out:
    for path in (CHECKPOINT, TRAIN_METADATA, REPORT, RUN_CONFIG):
        out.write(path, path.name)
print(json.dumps({key: report[key] for key in ('samples', 'accuracy', 'auc', 'f1', 'loss',
                                                'recall_ai', 'recall_human', 'threshold')}, indent=2))
print('ZIP:', archive)